# 第8章 面板数据与回归 —— 配套代码

对应正文 `book/part2/08-panel-regression.md`。

> **教学模拟数据说明**  
> 本 notebook 全部使用 `np.random.default_rng(42)` 生成的**教学演示面板**，  
> 模拟 100 家公司 × 10 年（2014—2023）共 1000 个观测。  
> 数据内置「已知真实系数」（$\beta_{\text{leverage}} = -0.15$，$\beta_{\text{size}} = 0.05$），  
> 便于直接验证 Pooled OLS / FE / RE 估计量能否还原真实参数。  
> **不代表任何真实公司。**实战请使用 CSMAR / Wind 财务数据库。

**本 notebook 演示内容**：
1. 生成教学模拟面板，展示 MultiIndex 结构
2. 描述性统计与个体效应可视化
3. Pooled OLS（混合回归）及偏误分析
4. 固定效应（FE）：个体效应 + 双向效应
5. 随机效应（RE）
6. 三种模型系数对比表
7. Hausman 检验（手动实现）
8. 普通标准误 vs 聚类标准误对比
9. 可视化：个体效应分布、拟合效果
10. 习题参考解答（可运行代码）

> **提示**：先运行 Cell 1 生成面板数据，其余格均依赖该格输出。

In [ ]:
# Cell 1：环境初始化与教学模拟面板生成
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from fds import set_chinese_font

set_chinese_font()

# ============================================================
# 面板参数设置
# ============================================================
N = 100          # 公司数
T = 10           # 年数（2014—2023）
SEED = 42
rng = np.random.default_rng(SEED)

# 真实系数（验证用）
BETA_LEV  = -0.15    # 杠杆率对ROA的真实效应
BETA_SIZE =  0.05    # 规模对ROA的真实效应
INTERCEPT =  0.08    # 截距（整体均值ROA≈8%）

years = list(range(2014, 2024))   # 10年
firms = [f'FIRM{i:03d}' for i in range(1, N+1)]

# ============================================================
# 生成个体固定效应（公司层面，不可观测，与杠杆相关）
#   alpha_i ~ N(0, 0.04^2)
#   设计：高alpha_i公司（盈利能力强）倾向于低杠杆
#   → alpha_i 与 leverage 负相关（ρ ≈ -0.5）
# ============================================================
alpha_i = rng.normal(0, 0.04, N)    # 个体固定效应（管理层能力等）

# ============================================================
# 生成时间固定效应（年度宏观冲击，所有公司共享）
# ============================================================
lambda_t = rng.normal(0, 0.015, T)  # 时间固定效应（GDP增速、货币政策等）

# ============================================================
# 生成协变量
# ============================================================
rows = []
for i, firm in enumerate(firms):
    # 公司杠杆率均值（与个体效应负相关，ρ≈-0.5）
    lev_base = 0.40 - 1.5 * alpha_i[i] + rng.normal(0, 0.05)
    lev_base = np.clip(lev_base, 0.05, 0.90)
    # 公司规模基础
    size_base = rng.uniform(20, 26)

    for t_idx, year in enumerate(years):
        leverage = lev_base + rng.normal(0, 0.04)
        leverage = np.clip(leverage, 0.05, 0.90)
        size = size_base + rng.normal(0, 0.3)
        # 误差项（含序列相关成分，模拟现实中的公司特定冲击）
        eps = rng.normal(0, 0.02)
        # 真实 ROA
        roa = (INTERCEPT
               + BETA_LEV * leverage
               + BETA_SIZE * (size - 23)  # 中心化
               + alpha_i[i]
               + lambda_t[t_idx]
               + eps)
        rows.append({
            'firm': firm,
            'year': year,
            'roa': roa,
            'leverage': leverage,
            'size': size - 23,   # 规模（已中心化）
            'true_alpha': alpha_i[i],
            'true_lambda': lambda_t[t_idx],
        })

# ============================================================
# 构造标准面板 DataFrame（MultiIndex: firm × year）
# ============================================================
df = pd.DataFrame(rows)
df = df.set_index(['firm', 'year'])

print(f'面板数据维度：{df.shape}')
print(f'个体数（N）：{N}，时期数（T）：{T}，总观测：{N*T}')
print(f'\nMultiIndex 结构（前5行）：')
df[['roa', 'leverage', 'size']].head()

## 8.2 面板结构检验与描述性统计

检查面板是否平衡，并对核心变量做描述性统计。

In [ ]:
# Cell 2：面板结构检验与描述统计

# 平衡性检验
obs_per_firm = df.groupby('firm').size()
is_balanced = (obs_per_firm == T).all()
print(f'是否平衡面板：{is_balanced}')
print(f'每家公司观测数：均为 {obs_per_firm.mean():.0f} 年')

print('\n核心变量描述性统计：')
desc = df[['roa', 'leverage', 'size']].describe().round(4)
desc.index.name = '统计量'
print(desc)

# 组间（between）变异 vs 组内（within）变异
print('\n组间 vs 组内变异分解（std）：')
for col in ['roa', 'leverage', 'size']:
    firm_mean = df[col].groupby('firm').transform('mean')
    between_std = df[col].groupby('firm').mean().std()  # 公司均值的标准差
    within_std  = (df[col] - firm_mean).std()            # 组内偏差的标准差
    print(f'  {col:10s}: between={between_std:.4f}, within={within_std:.4f}, '
          f'组间/组内={between_std/within_std:.2f}')

## 8.3 个体效应与杠杆率的相关性可视化

直觉验证：高个体效应（盈利能力强）的公司是否真的有更低的杠杆率？  
这正是 Pooled OLS 产生偏误的根源。

In [ ]:
# Cell 3：个体效应分布 + 与杠杆率的相关性散点

# 计算每家公司的平均值
firm_stats = df.groupby('firm').agg(
    avg_roa=('roa', 'mean'),
    avg_lev=('leverage', 'mean'),
    true_alpha=('true_alpha', 'first')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 左：个体效应分布
axes[0].hist(firm_stats['true_alpha'], bins=20, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='crimson', lw=2, linestyle='--')
axes[0].set_title('个体固定效应 $\\alpha_i$ 分布\n（不可观测，教学模拟）', fontsize=11)
axes[0].set_xlabel('$\\alpha_i$')
axes[0].set_ylabel('公司数量')

# 中：alpha_i vs 平均杠杆率
axes[1].scatter(firm_stats['true_alpha'], firm_stats['avg_lev'],
                alpha=0.6, s=30, color='darkorange')
rho = np.corrcoef(firm_stats['true_alpha'], firm_stats['avg_lev'])[0,1]
axes[1].set_title(f'$\\alpha_i$ vs 平均杠杆率\n相关系数 ρ = {rho:.3f}', fontsize=11)
axes[1].set_xlabel('个体效应 $\\alpha_i$（真实值）')
axes[1].set_ylabel('公司平均杠杆率')

# 右：alpha_i vs 平均ROA
axes[2].scatter(firm_stats['true_alpha'], firm_stats['avg_roa'],
                alpha=0.6, s=30, color='seagreen')
rho2 = np.corrcoef(firm_stats['true_alpha'], firm_stats['avg_roa'])[0,1]
axes[2].set_title(f'$\\alpha_i$ vs 平均ROA\n相关系数 ρ = {rho2:.3f}', fontsize=11)
axes[2].set_xlabel('个体效应 $\\alpha_i$（真实值）')
axes[2].set_ylabel('公司平均ROA')

plt.tight_layout()
plt.suptitle('个体固定效应与协变量的关系（教学模拟数据）', y=1.02, fontsize=12)
plt.show()

print(f'alpha_i 与杠杆率相关系数：{rho:.3f}（设计值约 -0.5）')
print(f'alpha_i 与ROA相关系数：{rho2:.3f}（应接近 1，因直接影响ROA）')
print('→ alpha_i 与 leverage 负相关，Pooled OLS 会混淆两者效应，产生偏误')

## 8.4 混合 OLS（Pooled OLS）

忽略面板结构，把所有观测当截面数据处理。  
预期：系数偏离真实值（偏误），标准误低估（t 虚高）。

> **真实系数**：$\beta_{\text{leverage}} = -0.15$，$\beta_{\text{size}} = 0.05$

In [ ]:
# Cell 4：Pooled OLS

y = df['roa']
X = sm.add_constant(df[['leverage', 'size']])

# linearmodels PooledOLS 需要 MultiIndex 数据
res_pooled = PooledOLS(y, X).fit()

print('=' * 55)
print('Pooled OLS 估计结果')
print('=' * 55)
print(res_pooled.summary.tables[1])
print(f'\nR²: {res_pooled.rsquared:.4f}')

print('\n=== 与真实值对比 ===')
coef_pooled = res_pooled.params
print(f'leverage 系数: 估计={coef_pooled["leverage"]:+.4f}, 真实={BETA_LEV:+.4f}, '
      f'偏误={coef_pooled["leverage"]-BETA_LEV:+.4f}')
print(f'size    系数: 估计={coef_pooled["size"]:+.4f}, 真实={BETA_SIZE:+.4f}, '
      f'偏误={coef_pooled["size"]-BETA_SIZE:+.4f}')
print('\n→ Pooled OLS leverage 系数偏误来自：alpha_i（能力强→低杠杆+高ROA）混入误差项')

## 8.5 固定效应模型（FE）

### 8.5.1 个体固定效应

组内变换消去 $\alpha_i$，利用公司内部时间变化识别系数。  
预期：系数接近真实值。

In [ ]:
# Cell 5：个体固定效应（FE）—— 普通SE vs 聚类SE

# 注意：FE 模型中不加常数（linearmodels 会自动处理）
X_fe = df[['leverage', 'size']]

# 5a. 普通（同方差）标准误
res_fe_plain = PanelOLS(y, X_fe, entity_effects=True).fit(cov_type='unadjusted')

# 5b. 聚类标准误（按公司）
res_fe_cl = PanelOLS(y, X_fe, entity_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

print('=' * 55)
print('个体固定效应（FE）—— 聚类标准误')
print('=' * 55)
print(res_fe_cl.summary.tables[1])
print(f'\nR²（组内）: {res_fe_cl.rsquared:.4f}')

print('\n=== 与真实值对比 ===')
coef_fe = res_fe_cl.params
print(f'leverage: 估计={coef_fe["leverage"]:+.4f}, 真实={BETA_LEV:+.4f}, '
      f'偏误={coef_fe["leverage"]-BETA_LEV:+.4f}')
print(f'size:     估计={coef_fe["size"]:+.4f}, 真实={BETA_SIZE:+.4f}, '
      f'偏误={coef_fe["size"]-BETA_SIZE:+.4f}')

print('\n=== 普通SE vs 聚类SE 对比 ===')
se_plain = res_fe_plain.std_errors
se_cl    = res_fe_cl.std_errors
for v in ['leverage', 'size']:
    print(f'{v:10s}: 普通SE={se_plain[v]:.5f}, 聚类SE={se_cl[v]:.5f}, '
          f'倍数={se_cl[v]/se_plain[v]:.2f}x')

### 8.5.2 双向固定效应（Two-Way FE）

同时控制个体效应（公司特征）和时间效应（宏观冲击）。

In [ ]:
# Cell 6：双向固定效应（个体 + 时间）

res_fe2w = PanelOLS(y, X_fe, entity_effects=True, time_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

print('=' * 55)
print('双向固定效应（个体 + 时间）—— 聚类标准误')
print('=' * 55)
print(res_fe2w.summary.tables[1])
print(f'\nR²（组内）: {res_fe2w.rsquared:.4f}')

coef_fe2w = res_fe2w.params
print('\n=== 与真实值对比 ===')
print(f'leverage: 估计={coef_fe2w["leverage"]:+.4f}, 真实={BETA_LEV:+.4f}')
print(f'size:     估计={coef_fe2w["size"]:+.4f}, 真实={BETA_SIZE:+.4f}')
print('\n→ 控制时间效应后，系数更接近真实值（消除了宏观因素混淆）')

## 8.6 随机效应模型（RE）

将 $\alpha_i$ 视为随机误差的一部分，用 GLS 估计。  
效率比 FE 高，但要求 $\alpha_i$ 与 $X$ 不相关（本例设计上违反了此假设）。

In [ ]:
# Cell 7：随机效应（RE）

X_re = sm.add_constant(df[['leverage', 'size']])
res_re = RandomEffects(y, X_re).fit()

print('=' * 55)
print('随机效应（RE）估计结果')
print('=' * 55)
print(res_re.summary.tables[1])

coef_re = res_re.params
print('\n=== 与真实值对比 ===')
print(f'leverage: 估计={coef_re["leverage"]:+.4f}, 真实={BETA_LEV:+.4f}')
print(f'size:     估计={coef_re["size"]:+.4f}, 真实={BETA_SIZE:+.4f}')
print('\n→ RE 系数介于 Pooled OLS 和 FE 之间（本例中RE违反假设，有偏）')

## 8.7 三种模型系数对比表

一张表汇总 Pooled OLS / FE(个体) / FE(双向) / RE 的系数、标准误与偏误。

In [ ]:
# Cell 8：四种模型系数对比表

models = {
    'Pooled OLS':      res_pooled,
    'FE（个体）':      res_fe_cl,
    'FE（双向）':      res_fe2w,
    'RE':              res_re,
}

rows_cmp = []
for name, res in models.items():
    p = res.params
    se = res.std_errors
    lev_key = 'leverage'
    sz_key  = 'size'
    rows_cmp.append({
        '模型':          name,
        'β_leverage':   round(p[lev_key], 4),
        'SE(lev)':      round(se[lev_key], 4),
        '偏误(lev)':    round(p[lev_key] - BETA_LEV, 4),
        'β_size':       round(p[sz_key], 4),
        'SE(size)':     round(se[sz_key], 4),
        '偏误(size)':   round(p[sz_key] - BETA_SIZE, 4),
    })

df_cmp = pd.DataFrame(rows_cmp).set_index('模型')

print('四种模型系数对比（真实值：leverage=-0.15，size=0.05）')
print('=' * 70)
print(df_cmp.to_string())
print('=' * 70)
print()
print('结论：')
print('  1. FE（个体）和FE（双向）系数最接近真实值')
print('  2. Pooled OLS 偏误最大（alpha_i 混淆了杠杆效应）')
print('  3. RE 系数介于两者之间（被偏误的组间信息拉偏）')

## 8.8 Hausman 检验

检验 $H_0$：$\text{Cov}(\mathbf{x}_{it}, \alpha_i) = 0$（RE 假设成立）  

本例中 $\alpha_i$ 与杠杆率刻意设计为负相关，期望 Hausman 检验**拒绝 $H_0$**（p < 0.05），从而支持选用 FE。

In [ ]:
# Cell 9：Hausman 检验（手动实现）
from scipy import stats as scipy_stats

# 使用同方差FE（Hausman检验基于传统SE）
res_fe_plain_full = PanelOLS(y, X_fe, entity_effects=True).fit(cov_type='unadjusted')
res_re_full = RandomEffects(y, X_re).fit()

# 对齐变量（FE不含常数，RE含常数）
common_vars = ['leverage', 'size']

b_fe = res_fe_plain_full.params[common_vars].values
b_re = res_re_full.params[common_vars].values

V_fe = res_fe_plain_full.cov[common_vars].loc[common_vars].values
V_re = res_re_full.cov[common_vars].loc[common_vars].values

diff = b_fe - b_re
V_diff = V_fe - V_re

# Hausman 统计量
try:
    V_diff_inv = np.linalg.inv(V_diff)
    H_stat = float(diff @ V_diff_inv @ diff)
    k = len(common_vars)
    p_val = 1 - scipy_stats.chi2.cdf(H_stat, df=k)
    print('Hausman 检验结果（手动实现）')
    print('=' * 50)
    print(f'统计量 H = {H_stat:.4f}')
    print(f'自由度  k = {k}')
    print(f'p 值      = {p_val:.6f}')
    print(f'5%临界值  = {scipy_stats.chi2.ppf(0.95, df=k):.4f}')
    print()
    if p_val < 0.05:
        print('结论：拒绝 H₀（p < 0.05）→ alpha_i 与 X 相关 → 选择 FE ✓')
    else:
        print('结论：不能拒绝 H₀（p ≥ 0.05）→ FE 与 RE 均一致 → 选效率更高的 RE')
except np.linalg.LinAlgError:
    print('V_diff 不可逆，尝试使用广义逆')
    H_stat = float(diff @ np.linalg.pinv(V_diff) @ diff)
    p_val = 1 - scipy_stats.chi2.cdf(H_stat, df=len(common_vars))
    print(f'H（广义逆）= {H_stat:.4f}, p = {p_val:.6f}')

print()
print('系数差异明细：')
for v, d, bfe, bre in zip(common_vars, diff, b_fe, b_re):
    print(f'  {v:10s}: FE={bfe:+.4f}, RE={bre:+.4f}, 差={d:+.4f}')

## 8.9 聚类标准误的重要性：可视化对比

同样的 FE 模型，普通标准误 vs 聚类标准误，差异有多大？

In [ ]:
# Cell 10：标准误对比可视化（普通 vs 聚类）

# 收集各模型的系数与置信区间
ci_data = []
for label, res, cov_label in [
    ('FE 普通SE',   res_fe_plain, '同方差'),
    ('FE 聚类SE',   res_fe_cl,   '按公司聚类'),
    ('FE双向 聚类SE', res_fe2w,  '按公司聚类'),
    ('Pooled OLS',  res_pooled,  '同方差'),
    ('RE',          res_re,      '标准GLS'),
]:
    for var in ['leverage', 'size']:
        if var in res.params.index:
            coef = res.params[var]
            se   = res.std_errors[var]
            ci_data.append({
                '模型': label, '变量': var,
                '系数': coef, 'SE': se,
                'CI下': coef - 1.96 * se,
                'CI上': coef + 1.96 * se,
            })

df_ci = pd.DataFrame(ci_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
model_order = ['Pooled OLS', 'FE 普通SE', 'FE 聚类SE', 'FE双向 聚类SE', 'RE']
colors = ['#d62728', '#1f77b4', '#2ca02c', '#17becf', '#ff7f0e']

for ax, var, true_val, title in [
    (axes[0], 'leverage', BETA_LEV, 'leverage 系数（真实值=-0.15）'),
    (axes[1], 'size',     BETA_SIZE, 'size 系数（真实值=+0.05）'),
]:
    sub = df_ci[df_ci['变量'] == var].set_index('模型').loc[model_order]
    y_pos = range(len(model_order))
    for i, (m, row) in enumerate(sub.iterrows()):
        ax.errorbar(row['系数'], i,
                    xerr=[[row['系数']-row['CI下']], [row['CI上']-row['系数']]],
                    fmt='o', color=colors[i], capsize=5, markersize=7,
                    label=m, elinewidth=2)
    ax.axvline(true_val, color='black', lw=2, linestyle='--', label=f'真实值={true_val}')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(model_order)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('系数（95% CI）')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('各模型系数估计与置信区间（教学模拟数据）', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('观察：')
print('  1. FE 普通SE 的置信区间最窄（低估了不确定性）')
print('  2. FE 聚类SE 的置信区间更宽（更诚实地反映序列相关）')
print('  3. Pooled OLS 系数偏离真实值（虚线）最远')

## 8.10 FE 估计值的精度分析

验证 FE（双向）能否在各种样本规模下还原真实系数。  
通过 Monte Carlo 模拟展示估计量的分布。

In [ ]:
# Cell 11：Monte Carlo 验证 FE 估计量的一致性

def generate_panel_and_estimate(N, T, seed):
    """生成模拟面板并返回 FE(个体) 对 leverage 的估计值"""
    rng_mc = np.random.default_rng(seed)
    alpha_mc = rng_mc.normal(0, 0.04, N)
    lambda_mc = rng_mc.normal(0, 0.015, T)
    rows_mc = []
    for i in range(N):
        lev_base = 0.40 - 1.5 * alpha_mc[i] + rng_mc.normal(0, 0.05)
        lev_base = np.clip(lev_base, 0.05, 0.90)
        size_base = rng_mc.uniform(20, 26)
        for t_idx in range(T):
            leverage = np.clip(lev_base + rng_mc.normal(0, 0.04), 0.05, 0.90)
            size = size_base + rng_mc.normal(0, 0.3)
            eps = rng_mc.normal(0, 0.02)
            roa = (INTERCEPT + BETA_LEV * leverage + BETA_SIZE * (size - 23)
                   + alpha_mc[i] + lambda_mc[t_idx] + eps)
            rows_mc.append({'firm': i, 'year': t_idx, 'roa': roa,
                            'leverage': leverage, 'size': size - 23})
    df_mc = pd.DataFrame(rows_mc).set_index(['firm', 'year'])
    y_mc = df_mc['roa']
    X_mc = df_mc[['leverage', 'size']]
    try:
        res_mc = PanelOLS(y_mc, X_mc, entity_effects=True).fit(cov_type='unadjusted')
        return res_mc.params['leverage']
    except Exception:
        return np.nan

# 运行 200 次 MC 模拟
N_MC = 200
estimates = [generate_panel_and_estimate(N=100, T=10, seed=i) for i in range(N_MC)]
estimates = np.array([e for e in estimates if not np.isnan(e)])

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(estimates, bins=30, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(BETA_LEV, color='crimson', lw=2.5, linestyle='--', label=f'真实值={BETA_LEV}')
ax.axvline(estimates.mean(), color='darkorange', lw=2, linestyle='-',
           label=f'MC均值={estimates.mean():.4f}')
ax.set_title(f'FE 估计量 $\\hat{{\\beta}}_{{leverage}}$ 的Monte Carlo分布 (n={len(estimates)}次)',
             fontsize=11)
ax.set_xlabel('$\\hat{\\beta}_{leverage}$')
ax.set_ylabel('概率密度')
ax.legend()
plt.tight_layout()
plt.show()

print(f'MC 均值: {estimates.mean():.4f}  （真实值: {BETA_LEV}）')
print(f'MC 标准差: {estimates.std():.4f}')
print(f'偏误: {estimates.mean()-BETA_LEV:.4f}  → FE 估计量对称地围绕真实值，近似无偏 ✓')

## 8.11 FE 拟合值与残差分析

In [ ]:
# Cell 12：拟合值与残差可视化

fitted = res_fe_cl.fitted_values.squeeze()   # DataFrame -> Series
residuals = res_fe_cl.resids.squeeze()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 左：实际值 vs 拟合值
axes[0].scatter(fitted, y, alpha=0.3, s=10, color='steelblue')
lim_min = min(fitted.min(), y.min()) * 0.95
lim_max = max(fitted.max(), y.max()) * 1.05
lim = [lim_min, lim_max]
axes[0].plot(lim, lim, 'r--', lw=1.5, label='45度线')
axes[0].set_xlabel('FE 拟合值')
axes[0].set_ylabel('实际 ROA')
axes[0].set_title('实际值 vs FE 拟合值')
axes[0].legend()

# 中：残差分布
axes[1].hist(residuals, bins=40, color='seagreen', alpha=0.7, edgecolor='white', density=True)
axes[1].set_title('FE 残差分布')
axes[1].set_xlabel('残差')
axes[1].set_ylabel('概率密度')

# 右：残差 vs 拟合值
axes[2].scatter(fitted, residuals, alpha=0.3, s=10, color='darkorange')
axes[2].axhline(0, color='black', lw=1.5, linestyle='--')
axes[2].set_xlabel('FE 拟合值')
axes[2].set_ylabel('残差')
axes[2].set_title('残差 vs 拟合值（无明显模式则良好）')

plt.suptitle('FE（个体效应，聚类SE）拟合诊断', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 8.12 个体固定效应的还原验证

固定效应估计器可以还原各公司的个体效应（$\hat{\alpha}_i$）。  
验证：估计出的 $\hat{\alpha}_i$ 与真实 $\alpha_i$ 的相关性应接近 1。

In [ ]:
# Cell 13：还原个体效应并与真实值对比

# 方法：利用 within 残差和组均值还原 alpha_hat
# alpha_i_hat = mean(y_i) - mean(X_i) * beta_hat
beta_hat = res_fe_cl.params.values  # [leverage, size]
X_vals = df[['leverage', 'size']].values

# 计算每个个体的组均值
df_est = df[['roa', 'leverage', 'size', 'true_alpha']].copy()
df_est['xbeta'] = X_vals @ beta_hat
df_est['alpha_hat'] = df_est['roa'] - df_est['xbeta']

# 组内均值即为 alpha_hat（因组内变换后均值为0）
alpha_comp = df_est.groupby('firm').agg(
    alpha_hat=('alpha_hat', 'mean'),
    alpha_true=('true_alpha', 'first')
)
# 中心化（FE估计量不识别总截距，只识别相对效应）
alpha_comp['alpha_hat_c'] = alpha_comp['alpha_hat'] - alpha_comp['alpha_hat'].mean()

corr = np.corrcoef(alpha_comp['alpha_true'], alpha_comp['alpha_hat_c'])[0,1]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(alpha_comp['alpha_true'], alpha_comp['alpha_hat_c'],
           alpha=0.6, s=30, color='steelblue')
lim_range = [alpha_comp['alpha_true'].min()*1.2, alpha_comp['alpha_true'].max()*1.2]
ax.plot(lim_range, lim_range, 'r--', lw=1.5, label='45度线（完美还原）')
ax.set_xlabel('真实 $\\alpha_i$')
ax.set_ylabel('FE 估计的 $\\hat{\\alpha}_i$（中心化）')
ax.set_title(f'个体效应还原验证\n相关系数 = {corr:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'真实 vs 估计 alpha_i 相关系数：{corr:.4f}')
print('→ 接近 1，说明 FE 成功识别了个体效应的相对大小 ✓')

## 8.13 本章小结

| 模型 | 系数一致性 | 个体效应假设 | 不随时间变量 | 推荐场景 |
|------|-----------|-------------|-------------|----------|
| Pooled OLS | $\alpha_i$ 与X无关时一致 | 完全忽略 | 可估 | 基准（不推荐单独汇报） |
| FE（个体） | **无条件一致** | 固定参数，可任意相关 | 无法估计 | 公司金融主回归 |
| FE（双向） | **无条件一致** | + 时间效应 | 无法估计 | 推荐标配 |
| RE | $\alpha_i \perp X$时一致 | 随机噪声 | 可估 | Hausman 不显著时 |

**黄金法则**：金融面板研究 = FE（双向）+ 聚类稳健标准误（按公司）

## 8.14 习题参考解答

以下代码对应正文第8.11节习题，可直接运行。

In [ ]:
# === 习题8.1：面板结构识别 ===
print('=' * 55)
print('习题 8.1：面板结构识别')
print('=' * 55)

# (a) 平衡性
obs_per_firm = df.groupby('firm').size()
is_bal = (obs_per_firm.nunique() == 1)
print(f'(a) 是否平衡面板：{is_bal}  |  观测数分布：{obs_per_firm.describe()[["min","max"]].to_dict()}')

# (b) MultiIndex 已构造
print(f'(b) MultiIndex 层级：{df.index.names}')

# (c) 每家公司观测年数
print(f'(c) 每家公司观测年数分布：')
print(obs_per_firm.value_counts().sort_index().to_dict())

# 模拟非平衡面板（随机删除 15% 数据）
rng_ex = np.random.default_rng(99)
mask = rng_ex.random(len(df)) > 0.15
df_unbal = df[mask].copy()
obs_unbal = df_unbal.groupby('firm').size()
print(f'\n模拟非平衡面板：{len(df_unbal)} 观测，每公司年数范围 [{obs_unbal.min()}, {obs_unbal.max()}]')

In [ ]:
# === 习题8.2：FE 组内变换推导（数值验证）===
print('=' * 55)
print('习题 8.2：FE 组内变换数值验证')
print('=' * 55)

# 取前3家公司（N=3, T=10）验证组内变换
demo_firms = firms[:3]
df_demo = df.loc[demo_firms]

# (a) 组内变换
df_demo_within = df_demo[['roa', 'leverage', 'size']].copy()
for col in ['roa', 'leverage', 'size']:
    mean_col = df_demo_within[col].groupby('firm').transform('mean')
    df_demo_within[f'{col}_within'] = df_demo_within[col] - mean_col

print('(a) 组内变换后各公司均值（应均为0）：')
print(df_demo_within[['roa_within','leverage_within','size_within']]
      .groupby('firm').mean().round(10))

# (c) 不随时间变化的变量：组内变换后为0
df_demo_within['constant_x'] = 1.0  # 不随时间变化
mean_const = df_demo_within['constant_x'].groupby('firm').transform('mean')
df_demo_within['constant_within'] = df_demo_within['constant_x'] - mean_const
print(f'\n(c) 不随时间变化的变量经组内变换后方差：{df_demo_within["constant_within"].var():.2e}')
print('  → 完全消失，FE 无法估计不随时间变化的效应 ✓')

In [ ]:
# === 习题8.3：Pooled OLS 偏误方向分析 ===
print('=' * 55)
print('习题 8.3：Pooled OLS 偏误方向分析')
print('=' * 55)

b_pool = res_pooled.params['leverage']
b_fe_val = res_fe_cl.params['leverage']

print(f'(a) 预测偏误方向：')
print(f'    alpha_i 与 leverage 负相关（高能力 → 低杠杆）')
print(f'    alpha_i 对 ROA 正影响')
print(f'    → Pooled OLS 会低估杠杆负效应的绝对值（即 |β̂_OLS| < |β_true|）')
print()
print(f'(b) 数值验证：')
print(f'    Pooled OLS β_leverage = {b_pool:+.4f}')
print(f'    FE β_leverage         = {b_fe_val:+.4f}')
print(f'    真实值                 = {BETA_LEV:+.4f}')
print()
print(f'(c) 偏误大小：Pooled OLS - FE = {b_pool - b_fe_val:+.4f}')
print(f'    Pooled OLS - 真实值        = {b_pool - BETA_LEV:+.4f}')
direction = '低估负效应（|β̂| 偏小）' if b_pool > BETA_LEV else '高估负效应（|β̂| 偏大）'
print(f'    偏误方向：{direction}')

In [ ]:
# === 习题8.4：手动 Hausman 检验（单变量版）===
print('=' * 55)
print('习题 8.4：手动 Hausman 检验（leverage 单变量）')
print('=' * 55)

b_fe_lev  = res_fe_plain_full.params['leverage']
b_re_lev  = res_re_full.params['leverage']
var_fe_lev = res_fe_plain_full.cov.loc['leverage', 'leverage']
var_re_lev = res_re_full.cov.loc['leverage', 'leverage']

diff_lev = b_fe_lev - b_re_lev
var_diff_lev = var_fe_lev - var_re_lev

print(f'β_FE  = {b_fe_lev:.6f},  Var(β_FE)  = {var_fe_lev:.8f}')
print(f'β_RE  = {b_re_lev:.6f},  Var(β_RE)  = {var_re_lev:.8f}')
print(f'差值  = {diff_lev:.6f},  Var差      = {var_diff_lev:.8f}')

if var_diff_lev > 0:
    H_manual = diff_lev**2 / var_diff_lev
    p_manual = 1 - scipy_stats.chi2.cdf(H_manual, df=1)
    crit_val = scipy_stats.chi2.ppf(0.95, df=1)
    print(f'\nH = {H_manual:.4f}')
    print(f'χ²(1) 5%临界值 = {crit_val:.4f}')
    print(f'p 值 = {p_manual:.6f}')
    conclusion = '拒绝H₀ → 选FE ✓' if p_manual < 0.05 else '不拒绝H₀ → FE与RE系数无显著差异'
    print(f'结论：{conclusion}')
else:
    print(f'Var差 = {var_diff_lev:.8f} < 0，Hausman 统计量分母非正（样本有限情形）')
    print('建议使用全变量版 Hausman 检验（Cell 9）')

In [ ]:
# === 习题8.5：双向FE vs 单向FE 时间效应检验 ===
print('=' * 55)
print('习题 8.5：双向FE 时间效应显著性')
print('=' * 55)

print('(a) 单向FE vs 双向FE 系数对比：')
for var in ['leverage', 'size']:
    b1 = res_fe_cl.params[var]
    b2 = res_fe2w.params[var]
    print(f'  {var:10s}: 单向FE={b1:+.4f}, 双向FE={b2:+.4f}, 差={b2-b1:+.4f}')

print()
print('(b) 为何时间效应影响估计：')
print('    宏观经济冲击同时影响所有公司的ROA（宏观景气↑→全行业ROA↑）')
print('    若宏观因素也影响杠杆决策（景气好→去杠杆），不控制时间效应')
print('    会把时间趋势的影响错误归因到杠杆系数上。')

print()
print('(c) 时间效应 F 检验（来自双向FE输出）：')
print(res_fe2w.summary.tables[0])

# 验证：时间效应估计值
# 用辅助回归：在FE(个体)基础上加年度虚拟变量
df_aug = df.reset_index().copy()
year_dummies = pd.get_dummies(df_aug['year'], prefix='yr', drop_first=True).astype(float)
df_aug = pd.concat([df_aug, year_dummies], axis=1).set_index(['firm', 'year'])

yd_cols = [c for c in df_aug.columns if c.startswith('yr_')]
X_aug = df_aug[['leverage', 'size'] + yd_cols]
res_aug = PanelOLS(df_aug['roa'], X_aug, entity_effects=True).fit(cov_type='clustered', cluster_entity=True)

print('\n年度虚拟变量系数（双向FE，以2014年为基准）：')
yr_coefs = res_aug.params[yd_cols]
yr_pvals = res_aug.pvalues[yd_cols]
yr_df = pd.DataFrame({'系数': yr_coefs.round(4), 'p值': yr_pvals.round(4)})
yr_df.index = [c.replace('yr_', '') for c in yd_cols]
yr_df.index.name = '年份'
print(yr_df.to_string())
print('\n真实时间效应（模拟设定值）：', np.round(lambda_t, 4))